## Path setup

In [8]:
# This notebook lives on notebooks/
# Code lives on src/

# Import path
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

# Import project modules
import config
import preprocessing

# Other imports
import pandas as pd
import numpy as np
import joblib       # For saving fitted objects

print("Imports OK. Project root:", config.PROJECT_ROOT)

Imports OK. Project root: /home/koala/lab/adversec


## Load processed data and preprocess

In [6]:
# Load the saved train and test sets from Stage 1
train_dup = pd.read_csv(config.PROCESSED_DIR / "ciciov2024_train_dup.csv")
test = pd.read_csv(config.PROCESSED_DIR / "ciciov2024_test.csv")

print("Loaded:")
print(f"    train_dup   : {train_dup.shape}")
print(f"    test        : {test.shape}\n")

# Encode labels: fit on train, apply to both. Returns interger arrays + encoder
y_train, y_test, label_encoder = preprocessing.encode_labels(train_dup, test)

# Scale the 9 features to [0,1]: fit on train, apply to both. Returns arrays + scaler
X_train, X_test, scaler = preprocessing.scale_features(train_dup, test, config.FEATURE_COLUMNS)

print(f"\nX_train shape     : {X_train.shape} (rows, features)")
print(f"X_test shape        : {X_test.shape}")
print(f"y_train shape       : {y_train.shape}")
print(f"y_test shape        : {y_test.shape}")

Loaded:
    train_dup   : (3838, 10)
    test        : (718, 10)

Label mapping:
    0 -> DoS
    1 -> benign
    2 -> spoofing-GAS
    3 -> spoofing-RPM
    4 -> spoofing-SPEED
    5 -> spoofing-STEERING_WHEEL

X_train shape     : (3838, 9) (rows, features)
X_test shape        : (718, 9)
y_train shape       : (3838,)
y_test shape        : (718,)


## Verify scaling and inspect test-set range

In [7]:
# Train was used to fit the scaler, so by definition its features sit in [0,1]
print("TRAIN feature range:")
print(f"    min     : {X_train.min():.4f}")
print(f"    max     : {X_train.max():.4f}")

# Test was only tansformed with train's min/max
# Values the training never saw can land slightly outside [0,1]
print("\nTEST feature range:")
print(f"    min     : {X_test.min():.4f}")
print(f"    max     : {X_test.max():.4f}")

# Count test cells that fall outside [0,1] to gauge how much the test set explores terriroty beyond the training data.
below = (X_test < 0).sum()
above = (X_test > 1).sum()
total = X_test.size

print(f"\nTest cells below 0: {below}    above 1: {above}   "
      f"({(below + above) / total * 100:.2f}% of {total} cells))")

TRAIN feature range:
    min     : 0.0000
    max     : 1.0000

TEST feature range:
    min     : -0.0045
    max     : 1.0217

Test cells below 0: 1    above 1: 1   (0.03% of 6462 cells))


## Save the processed arrays and the fitted objects

In [9]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save the 4 arrays in one compressed .npz file
arrays_path = config.PROCESSED_DIR / "stage2_arrays.npz"
np.savez_compressed(
    arrays_path,
    X_train = X_train,
    y_train = y_train,
    X_test = X_test,
    y_test = y_test,
)

# Save the fitted encoder and scaler.
# They currently hold the learned state. So they must travel with the arrays to stay consistent.
encoder_path = config.PROCESSED_DIR / "label_encoder.joblib"
scaler_path = config.PROCESSED_DIR / "feature_scaler.joblib"
joblib.dump(label_encoder, encoder_path)
joblib.dump(scaler, scaler_path)

print("Saved:")
for p in (arrays_path, encoder_path, scaler_path):
    print(" ", p)

Saved:
  /home/koala/lab/adversec/datasets/processed/stage2_arrays.npz
  /home/koala/lab/adversec/datasets/processed/label_encoder.joblib
  /home/koala/lab/adversec/datasets/processed/feature_scaler.joblib


## Stage 2 Summary: Preprocessing

Turns the cleaned CSVs from Stage 1 into model-ready arrays.

### What happened

- **Label encoding:** `true_class` strings mapped to integers, fitted on train only.
  Mapping (note: ASCII sort puts capital "DoS" before lowercase "benign"):

  | int | class |
  |---|---|
  | 0 | DoS |
  | 1 | benign |
  | 2 | spoofing-GAS |
  | 3 | spoofing-RPM |
  | 4 | spoofing-SPEED |
  | 5 | spoofing-STEERING_WHEEL |

- **Feature scaling:** all 9 features scaled to [0,1] with `MinMaxScaler`, **fit on
  train only** then applied to test (no leakage). This [0,1] space doubles as the
  coordinate system for adversarial perturbations later.

- **Test range check:** test features land in [-0.0045, 1.0217] — only 2 of 6,462
  cells fall outside [0,1] (0.03%), confirming the held-out signatures occupy nearly
  the same feature region as training. Values left unclipped (an honest signal).

### Saved artifacts

- `datasets/processed/stage2_arrays.npz` — X_train (3838×9), y_train, X_test (718×9), y_test
- `datasets/processed/label_encoder.joblib` — fitted encoder (int ↔ class name)
- `datasets/processed/feature_scaler.joblib` — fitted scaler (for consistent transforms + adversarial space)

### ID handling note

`ID` was scaled as a continuous feature consistent with prior CICIoV2024 work, rather
than one-hot encoded, to preserve a continuous perturbation space for adversarial
generation. (Limitation noted for the methodology chapter.)